# Notebook 7: FPGA Hardware Frequency Filter & High-Fidelity IFFT Verification

This notebook verifies the **FPGA real-time Frequency-Domain Filter and Inverse FFT (IFFT)** pipeline with **high-accuracy amplitude reconstruction** (`v1.6.0-rc1`).

### 🔬 Hardware Processing Pipeline:
$$\text{Raw Audio } x[n] \xrightarrow{\text{xfft\_0}} \text{Complex Spectrum } X[k] \xrightarrow{\text{axis\_spectral\_mask}} Y[k] \xrightarrow{\text{xfft\_1 (IFFT)}} \text{Filtered Audio } y[n]$$

### Verification Highlights:
1. 📊 **Synchronous 3-DMA Streaming:** Capture Raw Time (DMA 0), Filtered Time (DMA 2), and Spectrum (DMA 1) simultaneously in <2 ms.
2. 🎯 **High-Fidelity Amplitude Fidelity:** Calibrated amplitude matching between input and IFFT output ($V_{\text{pp, filt}} \approx V_{\text{pp, raw}}$).
3. 🎵 **Real-Time Bass Isolation:** Isolate sub-250 Hz bass from multi-tone chords with zero CPU load.
4. 🔊 **Direct Jupyter Audio Playback:** Listen to raw vs. filtered audio in-browser (`ol.play_audio()`).

## 1. System Setup & Permissions

In [ ]:
from pynq_oscilloscope import check_usb_permissions, OscilloscopeOverlay
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import time

check_usb_permissions()

# Load overlay and configure Audio Profile (50 kSPS, 1024-point FFT/IFFT)
ol = OscilloscopeOverlay()
ol.set_profile("audio")

print(f"✅ Overlay Loaded: Profile = {ol.current_profile} ({ol.sample_rate_hz/1e3:.1f} kSPS)")
print(f"   • Active Filter: {ol.filter}")

## 2. Inject Test Audio Tone (AD3 or Microphone Input)
Start generating a **100 Hz Sine Wave** (1.0V Amplitude, 1.65V Offset) on Channel 1 (W1 $\rightarrow$ A0).

In [ ]:
# Start 100 Hz Sine wave
ol.wavegen.start(shape="Sine", frequency=100.0, amplitude=1.0, offset=1.65)
time.sleep(0.5)
print("✅ Signal Generator active: 100 Hz Sine on A0 (1.0V Amp, 1.65V Offset)")

## 3. Baseline Test: Filter in Bypass Mode (Amplitude Fidelity Verification)
In bypass mode, the raw time waveform and the FPGA IFFT reconstructed waveform should match in amplitude ($V_{\text{pp}}$) and phase.

In [ ]:
# 1. Ensure filter is in Bypass
ol.filter.bypass()

# 2. Capture all 3 streams simultaneously
v_a0, v_a1, v_filt, freqs, mags = ol.capture_all()

vpp_raw = np.ptp(v_a0)
vpp_filt = np.ptp(v_filt)
error_pct = abs(vpp_filt - vpp_raw) / vpp_raw * 100.0

print(f"📊 Amplitude Fidelity Check:")
print(f"   • Raw Input A0 Vpp      : {vpp_raw:.2f} V")
print(f"   • IFFT Filtered Vpp     : {vpp_filt:.2f} V")
print(f"   • Amplitude Error       : {error_pct:.1f} %")

t_ms = np.linspace(0, (len(v_a0) / ol.sample_rate_hz) * 1000.0, len(v_a0))
show_pts = min(500, len(v_a0))

fig_bypass = go.Figure()
fig_bypass.add_scatter(x=t_ms[:show_pts], y=v_a0[:show_pts], mode="lines", line=dict(color="#00FFCC", width=1.5), name="Raw Input (A0)")
fig_bypass.add_scatter(x=t_ms[:show_pts], y=v_filt[:show_pts], mode="lines", line=dict(color="#FF007F", width=1.8, dash="dot"), name="IFFT Reconstructed")

fig_bypass.update_layout(
    title=f"<b>Bypass Fidelity Test: Raw vs. IFFT Output (Vpp Error: {error_pct:.1f}%)</b>",
    template="plotly_dark",
    xaxis_title="Time (ms)",
    yaxis_title="Voltage (V)",
    yaxis_range=[0, 3.3],
    height=400
)
fig_bypass.show()

## 4. Test 1: Real-Time FPGA Bassline Isolation (Lowpass Mode: 0 – 250 Hz)
Switch AD3 to a composite chord ($100\,\text{Hz}$ Bass + $2.5\,\text{kHz}$ Interference) and engage the hardware Lowpass filter.

In [ ]:
# Set AD3 to 2.5 kHz test tone
ol.wavegen.update_parameters(shape="Square", frequency=2500.0, amplitude=1.0)
time.sleep(0.3)

# Program FPGA filter: Lowpass < 250 Hz (Should reject 2.5 kHz tone completely!)
ol.filter.set_lowpass(cutoff_hz=250.0)
print(f"Active Filter: {ol.filter}")

v_a0, v_a1, v_bass, freqs, mags = ol.capture_all()

vpp_raw = np.ptp(v_a0)
vpp_bass = np.ptp(v_bass)
atten_db = 20.0 * np.log10(max(1e-4, vpp_bass) / max(1e-4, vpp_raw))

print(f"🔇 Rejection Performance:")
print(f"   • 2.5 kHz Raw Vpp      : {vpp_raw:.2f} V")
print(f"   • Filtered Bass Vpp    : {vpp_bass:.2f} V (Attenuated!)")
print(f"   • Measured Attenuation : {atten_db:.1f} dB")

fig_bass = make_subplots(
    rows=2, cols=1, vertical_spacing=0.15,
    subplot_titles=("<b>Raw 2.5 kHz Input Waveform (Channel 1 / A0)</b>", f"<b>FPGA-Filtered Output (Attenuated by {atten_db:.1f} dB)</b>")
)
fig_bass.add_scatter(x=t_ms[:show_pts], y=v_a0[:show_pts], mode="lines", line=dict(color="#00FFCC", width=1.5), row=1, col=1)
fig_bass.add_scatter(x=t_ms[:show_pts], y=v_bass[:show_pts], mode="lines", line=dict(color="#FF007F", width=2.0), row=2, col=1)

fig_bass.update_layout(template="plotly_dark", height=450, showlegend=False)
fig_bass.update_yaxes(title="Voltage (V)", range=[0, 3.3], row=1, col=1)
fig_bass.update_yaxes(title="Voltage (V)", range=[0, 3.3], row=2, col=1)
fig_bass.update_xaxes(title="Time (ms)", row=2, col=1)
fig_bass.show()

## 5. Test 2: Real-Time Highpass Filter (> 1 kHz Passband)
Switch filter to **Highpass Mode ($> 1\,\text{kHz}$)**. The $2.5\,\text{kHz}$ signal should pass through with full amplitude restored.

In [ ]:
# Program FPGA filter: Highpass > 1 kHz
ol.filter.set_highpass(cutoff_hz=1000.0)
print(f"Active Filter: {ol.filter}")

v_a0, v_a1, v_high, freqs, mags = ol.capture_all()

vpp_raw = np.ptp(v_a0)
vpp_high = np.ptp(v_high)

print(f"🎵 Highpass Performance:")
print(f"   • 2.5 kHz Raw Vpp      : {vpp_raw:.2f} V")
print(f"   • Highpass Passed Vpp  : {vpp_high:.2f} V (Restored!)")

fig_high = go.Figure()
fig_high.add_scatter(x=t_ms[:show_pts], y=v_a0[:show_pts], mode="lines", line=dict(color="#00FFCC", width=1.5), name="Raw Input")
fig_high.add_scatter(x=t_ms[:show_pts], y=v_high[:show_pts], mode="lines", line=dict(color="#FFA500", width=2.0, dash="dot"), name="Highpass Output")
fig_high.update_layout(
    title="<b>Highpass Test (> 1 kHz): Full Signal Recovery</b>",
    template="plotly_dark", height=400,
    xaxis_title="Time (ms)", yaxis_title="Voltage (V)", yaxis_range=[0, 3.3]
)
fig_high.show()

## 6. Auditory A/B Comparison: Raw vs. FPGA-Filtered Sound
Record 3 seconds of sound and listen to the raw vs. filtered audio streams using in-browser playback.

In [ ]:
ol.filter.set_highpass(cutoff_hz=1000.0)

print("🔊 1. Playing RAW Input Audio (Full Band):")
ol.play_audio(duration_sec=3.0, filtered=False)

print("🔊 2. Playing FPGA-FILTERED Audio (Highpass > 1 kHz):")
ol.play_audio(duration_sec=3.0, filtered=True)

## 7. Clean Hardware Shutdown

In [ ]:
ol.wavegen.stop()
ol.filter.bypass()
ol.close()
print("🔒 Hardware closed cleanly.")